# Nav1.5 MolProbity / clashscore analysis

The active analysis uses unchanged vanilla plus corrected masked v1 and corrected masked v2
cohorts. Retired masked CSVs are excluded at the shared data-access boundary, and no corrected
WT masked no-IFM replacement is currently staged; it is therefore not used as a fallback.
The corrected v1 and v2 cohorts remain distinct mask designs and are not pooled.

## Reproducible setup
Run from the repository root or channel directory. Dependencies: NumPy, pandas, SciPy, statsmodels, Matplotlib, seaborn, and Jupyter. MolProbity tables remain in the local downloaded directory. No upload or move is performed. Shared functions live in `shared/molprobity_analysis.py`, as in the existing analyses.

In [1]:
from pathlib import Path
import sys, os, json, platform, importlib.metadata
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'shared/data_access.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from shared import molprobity_analysis as mp
CHANNEL = 'nav15'
OUT = mp.output_dir(CHANNEL)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 24)
mp.apply_kv21_style()


## Data-access boundary
Set `VGIC_MOLPROBITY_ROOT` or `DATA_ROOT` to the downloaded directory. Future Hugging Face migration only needs to replace the loader with explicit paths through the existing `shared.data_access.resolve`; downstream analysis accepts the same DataFrame. Existing structural-QC and conformational inputs already use that resolver, including local/cache/offline behavior.

In [2]:
DATA_ROOT = os.environ.get('VGIC_MOLPROBITY_ROOT')
raw, inventory, total_csv_files = mp.load_molprobity(CHANNEL, DATA_ROOT)
inventory.to_csv(OUT/'tables/input_inventory.csv', index=False)
print(f'{total_csv_files} CSVs in downloaded directory; {len(inventory)} for this channel; {len(raw):,} rows')
display(inventory)


6 CSVs in downloaded directory; 6 for this channel; 36,000 rows


,csv_file,rows,sha256
0,Nav15_qqq_masked_v1_50pct_clashscores.csv,6000,65161cde539009805ddd88bed9d0534c921fd58be2701e...
1,Nav15_qqq_masked_v2_50pct_clashscores.csv,6000,501606366e44a418061bc92f744895f70be8a605896ff0...
2,Nav15_qqq_vanilla_clashscores.csv,6000,ce0ba1c55330b5d90a3bdd041db155f6a52388fc8d550c...
3,Nav15_wt_masked_v1_50pct_clashscores.csv,6000,2f7e7f915f1ae2744b4955943b76f1cce1c36f80ea96f8...
4,Nav15_wt_masked_v2_50pct_clashscores.csv,6000,2c40dfcc47545b628890a998e72e6829e1eeb09c7aa1aa...
5,Nav15_wt_vanilla_clashscores.csv,6000,7dd0c8ff7c5b7e6703215d4c6745c33a1cc7b5604ea6a4...


## Audit before plotting
The expected grid uses each ensemble’s recorded seed labels, AF models 1–5, r0–r10, and final. Missing whole trajectories/seeds are additionally checked against the existing model manifest. Raw input hashes are saved above. NaN recycle is expected only for final/base models; missing rank is allowed and reported.

In [3]:
data, audit = mp.audit_data(raw, CHANNEL, OUT)
display(audit)
display(pd.read_csv(OUT/'tables/missing_snapshots.csv'))
display(pd.read_csv(OUT/'tables/manifest_discrepancies.csv'))
display(pd.read_csv(OUT/'tables/row_anomalies.csv'))
cohorts = mp.select_cohorts(data, CHANNEL, OUT)
summary = pd.concat([mp.describe(frame, name) for name, frame in cohorts.items()], ignore_index=True)
summary.to_csv(OUT/'tables/descriptive_statistics.csv', index=False)
display(summary[(summary.cohort=='all_recorded') & summary.metric.eq('clashscore')].round(3))
display(pd.read_csv(OUT/'tables/selection_audit.csv'))


Audit: 36,000 rows, 0 failed, 0 invalid metric rows; 0 absent expected snapshots; 0 explicitly excluded rows.


,ensemble,rows,ok,ok_fraction,eligible,failed,missing_clashscore,nonfinite_clashscore,duplicate_names,duplicate_keys,missing_grid_snapshots,seeds,seed_min,seed_max,missing_rank,final_r10_pairs,final_r10_identical_metrics,manifest_rows,manifest_missing,manifest_extra
0,QQQ | masked,6000,6000,1.0,6000,0,0,0,0,0,0,100,33,132,0,500,500,6000,0,0
1,QQQ | masked v2,6000,6000,1.0,6000,0,0,0,0,0,0,100,33,132,0,500,500,6000,0,0
2,QQQ | vanilla,6000,6000,1.0,6000,0,0,0,0,0,0,100,17,116,0,500,500,6000,0,0
3,WT | masked,6000,6000,1.0,6000,0,0,0,0,0,0,100,33,132,0,500,500,6000,0,0
4,WT | masked v2,6000,6000,1.0,6000,0,0,0,0,0,0,100,33,132,0,500,500,6000,0,0
5,WT | vanilla,6000,6000,1.0,6000,0,0,0,0,0,0,100,43,142,0,500,500,6000,0,0


,ensemble,seed,af_model,snapshot


,ensemble,model_name,issue


,run_name,input_folder,model_name,rank,af_model,seed,recycle,snapshot,clashscore,n_bad_clashes,max_overlap,status,...,source_path,genotype,protocol,ensemble,csv_file,analysis_eligible,malformed_numeric,invalid_identity,duplicate_name,duplicate_key,invalid_metric,failed_status_or_exit


,cohort,ensemble,metric,N,n_seeds,median,Q1,Q3,IQR,mean,SD,minimum,maximum
0,all_recorded,QQQ | masked,clashscore,6000,100,30.900,29.16,35.532,6.372,33.063,5.458,25.57,73.12
3,all_recorded,QQQ | masked v2,clashscore,6000,100,30.460,28.26,35.022,6.762,32.343,5.495,24.46,62.35
6,all_recorded,QQQ | vanilla,clashscore,6000,100,28.680,27.51,32.240,4.730,30.423,4.513,23.91,51.49
9,all_recorded,WT | masked,clashscore,6000,100,30.540,29.15,35.520,6.370,32.821,5.278,25.87,51.78
12,all_recorded,WT | masked v2,clashscore,6000,100,30.205,28.09,35.250,7.160,32.081,5.374,24.22,54.74
15,all_recorded,WT | vanilla,clashscore,6000,100,28.640,27.42,32.415,4.995,30.400,4.520,23.51,53.16


,ensemble,selection,source,requested,matched,excluded_failed,excluded_by_structural_qc
0,QQQ | masked,3 Å convergence QC,nav15/rmsd_convergence_filtering/Nav15_rmsd_co...,4999,4999,0,1001
1,QQQ | masked v2,3 Å convergence QC,nav15/rmsd_convergence_filtering/Nav15_rmsd_co...,4999,4999,0,1001
2,QQQ | vanilla,3 Å convergence QC,nav15/rmsd_convergence_filtering/Nav15_rmsd_co...,5000,5000,0,1000
3,WT | masked,3 Å convergence QC,nav15/rmsd_convergence_filtering/Nav15_rmsd_co...,4998,4998,0,1002
4,WT | masked v2,3 Å convergence QC,nav15/rmsd_convergence_filtering/Nav15_rmsd_co...,5000,5000,0,1000
5,WT | vanilla,3 Å convergence QC,nav15/rmsd_convergence_filtering/Nav15_rmsd_co...,5000,5000,0,1000


## Statistical design and interpretation

The complete downloaded table is audited before filtering. `status != ok`, nonzero exit codes,
nonfinite/negative diagnostics and malformed identities are reported explicitly. Ambiguous identities
or mismatched manifests stop execution. Empty anomaly CSVs still retain their headers.

Every ensemble has up to 100 seeds × 5 AF models × 11 recycle snapshots plus a final/base file.
The final/base file has identical recorded MolProbity metrics to r10 for every available trajectory;
that is **metric equality, not a coordinate checksum**. To avoid double weighting those observations,
`all_recycles` excludes final copies. `final_only` is a separate sensitivity analysis.

The primary scientific comparison uses the repository's existing QC subset, independently of clashscore:
3 Å convergence for Cav1.2/Nav1.5 and the established structural/interface/alignment-QC distance
selection for Kv2.1. All raw distributions and excluded counts remain visible. QC-selected results
describe survivors, not all generated models. No new clashscore cutoff is used.

For each metric, average snapshots within seed/AF-model trajectory, then average AF models within seed.
This gives each retained seed equal weight and avoids treating recycles or the five AF models as
independent replicates. Point estimates are seed-balanced mean differences B−A; 95% percentile
bootstrap intervals use 3,000 seed draws. When seed-label sets match completely, nominal paired-seed
t tests and the existing `paired_common_seed_bootstrap` are used. Otherwise Welch tests compare all
retained seed summaries. Both independent-seed and common-label paired sensitivities are exported.
Shared numeric labels do **not** establish shared random-number streams; no seeds are renumbered,
and rank is never a pairing key. Common-label sensitivity may change the population when ranges differ.

BH correction covers all planned pairwise comparisons × all three metrics within each channel/cohort;
primary, independent, and common-label p-values form separate families. Supplemental cohort families
are exploratory. The seed rank-biserial effect is P(B>A)−P(B<A) across seed summaries, with ties zero;
it describes marginal ordering even for nominally paired tests. Model medians/IQRs are descriptive
snapshot summaries and are a different estimand from the seed-balanced mean contrast.

AF-model/snapshot-adjusted regressions provide a sensitivity check with standard errors clustered by
recorded seed. Rank is an output ordering variable, is absent for Kv2.1 L403A vanilla, and is inspected
descriptively rather than controlled as if it were a randomized covariate. These uncertainty estimates
describe computational sampling, not biological replication or a validated clinical effect.

Clashscore is a normalized global clash statistic ([Phenix glossary](https://www.phenix-online.org/version_docs/1.8.2-1296/glossary.htm)).
`n_bad_clashes` is an absolute count and `max_overlap` is a single extreme; they are complementary
diagnostics, not interchangeable biological measures. The local export does not record Phenix version,
hydrogen options, the atom denominator, or atom-level clash identities. Absolute validation grades,
cross-channel biological comparisons, and residue-specific attribution require additional provenance.


In [4]:
contrasts = mp.contrast_statistics(cohorts, CHANNEL, OUT)
primary = contrasts[(contrasts.cohort=='repository_QC') & contrasts.metric.eq('clashscore')]
display(primary[['A','B','N_A','N_B','n_seeds_A','n_seeds_B','common_seeds',
                 'common_model_snapshots','only_A_snapshots','only_B_snapshots',
                 'paired_seed_correlation','primary_test']])
display(primary[['A','B','median_difference','seed_mean_difference','CI_low','CI_high',
                 'seed_rank_biserial','q_BH','independent_q_BH','common_seed_difference',
                 'common_seed_CI_low','common_seed_CI_high','common_seed_q_BH']].round(4))


,A,B,N_A,N_B,n_seeds_A,n_seeds_B,common_seeds,common_model_snapshots,only_A_snapshots,only_B_snapshots,paired_seed_correlation,primary_test
27,WT | vanilla,WT | masked,5000,4998,100,100,90,4498,502,500,0.203528,Welch seed t
30,WT | vanilla,WT | masked v2,5000,5000,100,100,90,4500,500,500,0.302519,Welch seed t
33,QQQ | vanilla,QQQ | masked,5000,4999,100,100,84,4200,800,799,0.237797,Welch seed t
36,QQQ | vanilla,QQQ | masked v2,5000,4999,100,100,84,4199,801,800,0.222488,Welch seed t
39,WT | vanilla,QQQ | vanilla,5000,5000,100,100,74,3700,1300,1300,0.071035,Welch seed t
42,WT | masked,QQQ | masked,4998,4999,100,100,100,4997,1,2,0.026306,paired nominal seed t
45,WT | masked v2,QQQ | masked v2,5000,4999,100,100,100,4999,1,0,0.155378,paired nominal seed t
48,QQQ | masked,QQQ | masked v2,4999,4999,100,100,100,4998,1,1,0.376681,paired nominal seed t
51,WT | masked,WT | masked v2,4998,5000,100,100,100,4998,0,2,0.393010,paired nominal seed t


,A,B,median_difference,seed_mean_difference,CI_low,CI_high,seed_rank_biserial,q_BH,independent_q_BH,common_seed_difference,common_seed_CI_low,common_seed_CI_high,common_seed_q_BH
27,WT | vanilla,WT | masked,1.94,2.4744,2.4040,2.5429,1.0000,0.0000,0.0000,2.4893,2.4263,2.5565,0.0000
30,WT | vanilla,WT | masked v2,1.50,1.7800,1.7082,1.8502,1.0000,0.0000,0.0000,1.7844,1.7208,1.8487,0.0000
33,QQQ | vanilla,QQQ | masked,2.22,2.7172,2.6391,2.7980,1.0000,0.0000,0.0000,2.7343,2.6643,2.8082,0.0000
36,QQQ | vanilla,QQQ | masked v2,1.71,2.0326,1.9557,2.1071,1.0000,0.0000,0.0000,2.0314,1.9571,2.1031,0.0000
39,WT | vanilla,QQQ | vanilla,0.04,-0.0016,-0.0755,0.0730,0.0116,0.9659,0.9659,-0.0171,-0.1030,0.0650,0.6905
42,WT | masked,QQQ | masked,0.32,0.2412,0.1725,0.3125,0.4731,0.0000,0.0000,0.2412,0.1725,0.3125,0.0000
45,WT | masked v2,QQQ | masked v2,0.25,0.2510,0.1878,0.3192,0.4869,0.0000,0.0000,0.2510,0.1878,0.3192,0.0000
48,QQQ | masked,QQQ | masked v2,-0.51,-0.6845,-0.7425,-0.6260,-0.9322,0.0000,0.0000,-0.6845,-0.7425,-0.6260,0.0000
51,WT | masked,WT | masked v2,-0.44,-0.6943,-0.7484,-0.6399,-0.9429,0.0000,0.0000,-0.6943,-0.7484,-0.6399,0.0000


## Clashscore distributions and effect sizes
Violins retain every eligible individual snapshot with median and quartiles; no outliers are clipped. The paired raw/QC panels expose selection effects. The contrast figure shows seed-balanced differences and 95% bootstrap intervals; q-values are reported without significance stars. Colors, typography, thin outlines, light grids, and PDF/300-dpi PNG exports follow `shared.plotting` and existing conformational figures.

In [5]:
mp.plot_distributions(cohorts, CHANNEL, OUT)
plt.show()
mp.plot_effects(contrasts, CHANNEL, OUT)
plt.show()
display(Markdown(mp.interpretation(summary, contrasts, CHANNEL)))


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_78811/2804364640.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_78811/2804364640.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Results on the repository QC subset

Model medians describe retained snapshots. Differences and confidence intervals below use equal AF-model weights within seed and seed-level inference.
- **WT | masked versus WT | vanilla:** model medians 30.46 versus 28.52 (Δ +1.94); seed-balanced mean Δ +2.47, 95% CI [2.40, 2.54]; seed rank-biserial effect +1.00; BH q=2.15e-139.
- **WT | masked v2 versus WT | vanilla:** model medians 30.02 versus 28.52 (Δ +1.50); seed-balanced mean Δ +1.78, 95% CI [1.71, 1.85]; seed rank-biserial effect +1.00; BH q=5.28e-109.
- **QQQ | masked versus QQQ | vanilla:** model medians 30.78 versus 28.56 (Δ +2.22); seed-balanced mean Δ +2.72, 95% CI [2.64, 2.80]; seed rank-biserial effect +1.00; BH q=1.88e-139.
- **QQQ | masked v2 versus QQQ | vanilla:** model medians 30.27 versus 28.56 (Δ +1.71); seed-balanced mean Δ +2.03, 95% CI [1.96, 2.11]; seed rank-biserial effect +1.00; BH q=5.38e-119.
- **QQQ | vanilla versus WT | vanilla:** model medians 28.56 versus 28.52 (Δ +0.04); seed-balanced mean Δ -0.00, 95% CI [-0.08, 0.07]; seed rank-biserial effect +0.01; BH q=0.966.
- **QQQ | masked versus WT | masked:** model medians 30.78 versus 30.46 (Δ +0.32); seed-balanced mean Δ +0.24, 95% CI [0.17, 0.31]; seed rank-biserial effect +0.47; BH q=4.6e-09.
- **QQQ | masked v2 versus WT | masked v2:** model medians 30.27 versus 30.02 (Δ +0.25); seed-balanced mean Δ +0.25, 95% CI [0.19, 0.32]; seed rank-biserial effect +0.49; BH q=2.4e-10.
- **QQQ | masked v2 versus QQQ | masked:** model medians 30.27 versus 30.78 (Δ -0.51); seed-balanced mean Δ -0.68, 95% CI [-0.74, -0.63]; seed rank-biserial effect -0.93; BH q=6.41e-40.
- **WT | masked v2 versus WT | masked:** model medians 30.02 versus 30.46 (Δ -0.44); seed-balanced mean Δ -0.69, 95% CI [-0.75, -0.64]; seed rank-biserial effect -0.94; BH q=6.79e-43.

## Complementary steric diagnostics
These panels use the repository QC subset. Counts depend on structure size and atom handling; maximum overlap reports one extreme contact. Full numerical results and corrected comparisons for both metrics are retained alongside clashscore.

In [6]:
mp.plot_diagnostics(cohorts, CHANNEL, OUT)
plt.show()
display(summary[(summary.cohort=='repository_QC') & summary.metric.ne('clashscore')].round(3))
display(contrasts[(contrasts.cohort=='repository_QC') & contrasts.metric.ne('clashscore')][
    ['A','B','metric','median_difference','seed_mean_difference','CI_low','CI_high','seed_rank_biserial','q_BH']].round(4))


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_78811/1691443703.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,cohort,ensemble,metric,N,n_seeds,median,Q1,Q3,IQR,mean,SD,minimum,maximum
55,repository_QC,QQQ | masked,n_bad_clashes,4999,100,779.000,737.000,866.000,129.000,827.730,129.776,647.000,1387.000
56,repository_QC,QQQ | masked,max_overlap,4999,100,1.624,1.592,1.655,0.063,1.620,0.053,1.328,1.788
58,repository_QC,QQQ | masked v2,n_bad_clashes,4999,100,766.000,714.000,859.000,145.000,810.362,131.028,619.000,1229.000
59,repository_QC,QQQ | masked v2,max_overlap,4999,100,1.621,1.571,1.656,0.085,1.606,0.075,1.348,1.781
61,repository_QC,QQQ | vanilla,n_bad_clashes,5000,100,723.000,695.000,773.000,78.000,758.952,99.522,605.000,1176.000
62,repository_QC,QQQ | vanilla,max_overlap,5000,100,1.626,1.597,1.654,0.057,1.623,0.047,1.378,1.748
64,repository_QC,WT | masked,n_bad_clashes,4998,100,771.000,737.000,863.750,126.750,821.737,124.980,655.000,1256.000
65,repository_QC,WT | masked,max_overlap,4998,100,1.630,1.599,1.659,0.060,1.625,0.055,1.347,1.783
67,repository_QC,WT | masked v2,n_bad_clashes,5000,100,760.000,711.000,861.000,150.000,804.207,128.237,613.000,1234.000
68,repository_QC,WT | masked v2,max_overlap,5000,100,1.624,1.575,1.660,0.085,1.610,0.075,1.309,1.787


,A,B,metric,median_difference,seed_mean_difference,CI_low,CI_high,seed_rank_biserial,q_BH
28,WT | vanilla,WT | masked,n_bad_clashes,49.000,62.6311,60.8505,64.3654,1.0000,0.0000
29,WT | vanilla,WT | masked,max_overlap,0.000,-0.0016,-0.0036,0.0004,-0.1292,0.1338
31,WT | vanilla,WT | masked v2,n_bad_clashes,38.000,45.0566,43.2360,46.8317,1.0000,0.0000
32,WT | vanilla,WT | masked v2,max_overlap,-0.006,-0.0165,-0.0186,-0.0143,-0.8616,0.0000
34,QQQ | vanilla,QQQ | masked,n_bad_clashes,56.000,68.7732,66.8004,70.8215,1.0000,0.0000
35,QQQ | vanilla,QQQ | masked,max_overlap,-0.002,-0.0024,-0.0044,-0.0004,-0.1813,0.0283
37,QQQ | vanilla,QQQ | masked v2,n_bad_clashes,43.000,51.4554,49.5049,53.3378,1.0000,0.0000
38,QQQ | vanilla,QQQ | masked v2,max_overlap,-0.005,-0.0162,-0.0185,-0.0140,-0.8334,0.0000
40,WT | vanilla,QQQ | vanilla,n_bad_clashes,1.000,-0.1982,-2.0682,1.6875,-0.0006,0.8669
41,WT | vanilla,QQQ | vanilla,max_overlap,-0.004,-0.0040,-0.0058,-0.0023,-0.3852,0.0000


## Supplementary mask-design controls
Corrected masked v1 and v2 remain separate cohorts. The comparison below tests sensitivity to the two corrected mask designs; retired legacy masked data and the unavailable no-IFM replacement are not included.

In [7]:
mp.plot_nav_controls(cohorts, OUT)
plt.show()

/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_78811/2631131968.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Recycle, AF-model, seed, and rank sensitivity
The full snapshot trace includes final separately and all early-recycle tails. The final-only analysis does not substitute a last available snapshot for a missing final file. The adjusted regression controls AF-model and snapshot composition, with uncertainty clustered by seed; its observation-weighted estimand differs from the seed-balanced primary result.

In [8]:
mp.stratification(cohorts, CHANNEL, OUT)
plt.show()
adjusted = mp.adjusted_contrasts(cohorts, CHANNEL, OUT)
display(adjusted.round(4))
af_sensitivity = mp.af_model_sensitivity(cohorts, CHANNEL, OUT)
display(af_sensitivity.pivot(index=['A','B','mode'], columns='AF_model', values='seed_mean_difference').round(3))
display(contrasts[contrasts.metric.eq('clashscore')][
    ['cohort','A','B','seed_mean_difference','CI_low','CI_high','q_BH']].round(3))


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_78811/2647556288.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,A,B,adjusted_difference,CI_low,CI_high,p,q_BH
0,WT | vanilla,WT | masked,2.4752,2.4118,2.5386,0.0000,0.0000
1,WT | vanilla,WT | masked v2,1.7800,1.7166,1.8435,0.0000,0.0000
2,QQQ | vanilla,QQQ | masked,2.7178,2.6491,2.7864,0.0000,0.0000
3,QQQ | vanilla,QQQ | masked v2,2.0330,1.9657,2.1003,0.0000,0.0000
4,WT | vanilla,QQQ | vanilla,-0.0016,-0.0730,0.0698,0.9649,0.9649
5,WT | masked,QQQ | masked,0.2407,0.1680,0.3135,0.0000,0.0000
6,WT | masked v2,QQQ | masked v2,0.2515,0.1827,0.3204,0.0000,0.0000
7,QQQ | masked,QQQ | masked v2,-0.6847,-0.7449,-0.6245,0.0000,0.0000
8,WT | masked,WT | masked v2,-0.6955,-0.7515,-0.6395,0.0000,0.0000


AF_model                                               1      2      3      4  \
A              B               mode                                             
QQQ | masked   QQQ | masked v2 leave_AF_model_out -0.659 -0.914 -0.613 -0.654   
                               only_AF_model      -0.786  0.235 -0.969 -0.807   
QQQ | vanilla  QQQ | masked    leave_AF_model_out  2.164  2.525  2.741  3.082   
                               only_AF_model       4.930  3.486  2.622  1.257   
               QQQ | masked v2 leave_AF_model_out  1.505  1.610  2.128  2.428   
                               only_AF_model       4.145  3.721  1.653  0.451   
WT | masked    QQQ | masked    leave_AF_model_out  0.159  0.181  0.231  0.291   
                               only_AF_model       0.571  0.482  0.284  0.041   
               WT | masked v2  leave_AF_model_out -0.669 -1.003 -0.656 -0.631   
                               only_AF_model      -0.794  0.541 -0.847 -0.949   
WT | masked v2 QQQ | masked v2 leave_AF_model_out  0.169  0.270  0.273  0.268   
                               only_AF_model       0.579  0.176  0.162  0.184   
WT | vanilla   QQQ | vanilla   leave_AF_model_out  0.023 -0.029  0.059 -0.047   
                               only_AF_model      -0.102  0.109 -0.244  0.178   
               WT | masked     leave_AF_model_out  2.029  2.315  2.569  2.744   
                               only_AF_model       4.258  3.113  2.094  1.394   
               WT | masked v2  leave_AF_model_out  1.359  1.312  1.913  2.114   
                               only_AF_model       3.464  3.654  1.247  0.445   

AF_model                                               5  
A              B               mode                       
QQQ | masked   QQQ | masked v2 leave_AF_model_out -0.582  
                               only_AF_model      -1.096  
QQQ | vanilla  QQQ | masked    leave_AF_model_out  3.074  
                               only_AF_model       1.290  
               QQQ | masked v2 leave_AF_model_out  2.492  
                               only_AF_model       0.193  
WT | masked    QQQ | masked    leave_AF_model_out  0.345  
                               only_AF_model      -0.172  
               WT | masked v2  leave_AF_model_out -0.512  
                               only_AF_model      -1.423  
WT | masked v2 QQQ | masked v2 leave_AF_model_out  0.275  
                               only_AF_model       0.154  
WT | vanilla   QQQ | vanilla   leave_AF_model_out -0.015  
                               only_AF_model       0.051  
               WT | masked     leave_AF_model_out  2.715  
                               only_AF_model       1.513  
               WT | masked v2  leave_AF_model_out  2.202  
                               only_AF_model       0.090

,cohort,A,B,seed_mean_difference,CI_low,CI_high,q_BH
0,all_recycles,WT | vanilla,WT | masked,2.435,2.367,2.503,0.000
3,all_recycles,WT | vanilla,WT | masked v2,1.691,1.620,1.762,0.000
6,all_recycles,QQQ | vanilla,QQQ | masked,2.662,2.584,2.739,0.000
9,all_recycles,QQQ | vanilla,QQQ | masked v2,1.931,1.852,2.006,0.000
12,all_recycles,WT | vanilla,QQQ | vanilla,0.008,-0.067,0.082,0.865
15,all_recycles,WT | masked,QQQ | masked,0.235,0.170,0.301,0.000
18,all_recycles,WT | masked v2,QQQ | masked v2,0.248,0.182,0.320,0.000
21,all_recycles,QQQ | masked,QQQ | masked v2,-0.731,-0.794,-0.666,0.000
24,all_recycles,WT | masked,WT | masked v2,-0.744,-0.800,-0.689,0.000
27,repository_QC,WT | vanilla,WT | masked,2.474,2.404,2.543,0.000


## Focused interpretation
The following text is generated from the executed result tables so it stays synchronized with reruns.

In [9]:
display(Markdown(mp.focused_interpretation(summary, contrasts, CHANNEL, OUT)))

**Primary sequence versus mask comparison.** WT vanilla and QQQ vanilla medians are 28.52 and 28.56; seed-balanced QQQ−WT is -0.002 (95% CI -0.076 to +0.073). QQQ corrected masked v1 raises the median to 30.78, with a seed-balanced change of +2.717 (95% CI +2.639 to +2.798).

Corrected masked v2 versus corrected masked v1 changes the mean by -0.685 (95% CI -0.742 to -0.626).

The principal global difference follows mask design, not the QQQ substitution under vanilla. Maximum-overlap medians remain close across the ensembles; there is no unique catastrophic global-quality outlier among the retained conditions. This is a relative QC statement, not certification of atomistic quality in these unrelaxed models. The mask-related clashscore cost is useful alongside the pore-sampling figures; most alternative-mask results belong in supplementary QC.

**AF-model dependence.** The following ranges are effect magnitudes, not new hypothesis tests:

- WT masked−vanilla: individual AF-model mean differences range from +1.39 to +4.26; leaving out one AF model at a time gives +2.03 to +2.74.

- QQQ masked−vanilla: individual AF-model mean differences range from +1.26 to +4.93; leaving out one AF model at a time gives +2.16 to +3.08.

Original masking raises the mean within every AF-model stratum. The stratified tables and final-r10 sensitivities should accompany claims based on pooled means.

## Reproducibility record
All generated outputs are confined to this channel’s `dataExtra/molprobity/{tables,figures}`. Existing figures and analyses are not overwritten. Input hashes, missing-model lists, full descriptive/contrast tables, pairing diagnostics and sensitivity results accompany the figures.

In [10]:
record = {'channel': CHANNEL, 'total_downloaded_csvs': total_csv_files,
          'channel_csvs': len(inventory), 'rows': len(raw), 'python': platform.python_version(),
          'bootstrap_replicates': mp.BOOTSTRAPS, 'random_seed': mp.RANDOM_SEED,
          'packages': {p: importlib.metadata.version(p) for p in ['numpy','pandas','scipy','statsmodels','matplotlib','seaborn']}}
(OUT/'tables/run_metadata.json').write_text(json.dumps(record, indent=2)+chr(10))
print('Completed:', OUT.relative_to(ROOT))


Completed: nav15/dataExtra/molprobity
